In [1]:
import os

print("Kaggle input folders:")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level < 2:
        print(root)

        for file in files[:20]:
            print("   ", file)

Kaggle input folders:
/kaggle/input
/kaggle/input/competitions


In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("\nFOLDER:", root)

    for file in files[:20]:
        print("   ", file)


FOLDER: /kaggle/input

FOLDER: /kaggle/input/competitions

FOLDER: /kaggle/input/competitions/shl-hiring-assessment-2026

FOLDER: /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final
    sample_submission.csv
    train.csv
    test.csv

FOLDER: /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test
    audio_49.wav
    audio_90.wav
    audio_77.wav
    audio_66.wav
    audio_54.wav
    audio_42.wav
    audio_81.wav
    audio_72.wav
    audio_107.wav
    audio_79.wav
    audio_83.wav
    audio_88.wav
    audio_164.wav
    audio_60.wav
    audio_55.wav
    audio_204.wav
    audio_25.wav
    audio_112.wav
    audio_176.wav
    audio_132.wav

FOLDER: /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train
    audio_49.wav
    audio_663.wav
    audio_5059.wav
    audio_90.wav
    audio_581.wav
    audio_698.wav
    audio_77.wav
    audio_66.wav
    audio_555.wav
    audio_54.wav
    audio_237.wav
    audio_520.wav
    audio_765.wav
    aud

In [3]:
import os

# Find all CSV files
csv_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("CSV files found:\n")

for file in csv_files:
    print(file)

CSV files found:

/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv


In [ ]:
import pandas as pd

# Automatically locate the required files
train_path = [x for x in csv_files if x.endswith("train.csv")][0]
test_path = [x for x in csv_files if x.endswith("test.csv")][0]
submission_path = [x for x in csv_files if x.endswith("sample_submission.csv")][0]

# Load datasets
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(submission_path)

print("TRAIN:")
print(train.shape)
display(train.head())

print("\nTEST:")
print(test.shape)
display(test.head())

print("\nSAMPLE SUBMISSION:")
print(sample_submission.shape)
display(sample_submission.head())

In [ ]:
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nSubmission columns:")
print(sample_submission.columns.tolist())

In [ ]:
import os

wav_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".wav"):
            wav_files.append(os.path.join(root, file))

print("Total WAV files found:", len(wav_files))

print("\nFirst 10 WAV files:")
for file in wav_files[:10]:
    print(file)

In [ ]:
print("Training data:")
display(train.head(10))

print("\nTraining shape:")
print(train.shape)

print("\nLabel statistics:")
print(train["label"].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

sns.histplot(
    train["label"],
    bins=20,
    kde=True
)

plt.title("Distribution of Grammar Scores")
plt.xlabel("Grammar Score")
plt.ylabel("Number of Audio Samples")

plt.show()

In [ ]:
wav_lookup = {
    os.path.basename(path): path
    for path in wav_files
}

train["audio_path"] = train["filename"].map(wav_lookup)
test["audio_path"] = test["filename"].map(wav_lookup)

print("Training audio files found:",
      train["audio_path"].notna().sum(),
      "/", len(train))

print("Test audio files found:",
      test["audio_path"].notna().sum(),
      "/", len(test))

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Pick the first training audio file
audio_path = train.loc[0, "audio_path"]

# Load audio
audio, sample_rate = librosa.load(
    audio_path,
    sr=None
)

duration = len(audio) / sample_rate

print("Audio file:", train.loc[0, "filename"])
print("Grammar score:", train.loc[0, "label"])
print("Sample rate:", sample_rate)
print("Number of samples:", len(audio))
print("Duration:", round(duration, 2), "seconds")

In [ ]:
plt.figure(figsize=(14, 4))

librosa.display.waveshow(
    audio,
    sr=sample_rate
)

plt.title("Speech Audio Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")

plt.show()

In [ ]:
# Short-Time Fourier Transform
stft = librosa.stft(audio)

# Convert amplitude to decibels
spectrogram = librosa.amplitude_to_db(
    np.abs(stft),
    ref=np.max
)

plt.figure(figsize=(14, 5))

librosa.display.specshow(
    spectrogram,
    sr=sample_rate,
    x_axis="time",
    y_axis="hz"
)

plt.colorbar(format="%+2.0f dB")
plt.title("Speech Spectrogram")
plt.xlabel("Time")
plt.ylabel("Frequency")

plt.show()

In [ ]:
mfcc = librosa.feature.mfcc(
    y=audio,
    sr=sample_rate,
    n_mfcc=40
)

print("MFCC shape:", mfcc.shape)

In [ ]:
plt.figure(figsize=(14, 5))

librosa.display.specshow(
    mfcc,
    x_axis="time",
    sr=sample_rate
)

plt.colorbar()
plt.title("MFCC Features")
plt.xlabel("Time")
plt.ylabel("MFCC Coefficient")

plt.show()

In [ ]:
print("Minimum score:", train["label"].min())
print("Maximum score:", train["label"].max())
print("Average score:", train["label"].mean())
print("Median score:", train["label"].median())
print("Standard deviation:", train["label"].std())

In [ ]:
import numpy as np
import librosa
from tqdm.auto import tqdm

def extract_audio_features(file_path):
    """
    Extract fixed-length audio features from a WAV file.
    """

    # Load audio at 16 kHz
    y, sr = librosa.load(
        file_path,
        sr=16000,
        mono=True
    )

    features = []

    # --------------------------------------------------
    # 1. MFCC
    # --------------------------------------------------
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=40
    )

    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    # --------------------------------------------------
    # 2. MFCC Delta
    # --------------------------------------------------
    mfcc_delta = librosa.feature.delta(mfcc)

    features.extend(np.mean(mfcc_delta, axis=1))
    features.extend(np.std(mfcc_delta, axis=1))

    # --------------------------------------------------
    # 3. Mel Spectrogram
    # --------------------------------------------------
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=40
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    features.extend(np.mean(mel_db, axis=1))
    features.extend(np.std(mel_db, axis=1))

    # --------------------------------------------------
    # 4. Chroma
    # --------------------------------------------------
    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr
    )

    features.extend(np.mean(chroma, axis=1))
    features.extend(np.std(chroma, axis=1))

    # --------------------------------------------------
    # 5. Spectral Centroid
    # --------------------------------------------------
    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )

    features.append(np.mean(centroid))
    features.append(np.std(centroid))

    # --------------------------------------------------
    # 6. Spectral Bandwidth
    # --------------------------------------------------
    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr
    )

    features.append(np.mean(bandwidth))
    features.append(np.std(bandwidth))

    # --------------------------------------------------
    # 7. Spectral Rolloff
    # --------------------------------------------------
    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=sr
    )

    features.append(np.mean(rolloff))
    features.append(np.std(rolloff))

    # --------------------------------------------------
    # 8. Zero Crossing Rate
    # --------------------------------------------------
    zcr = librosa.feature.zero_crossing_rate(y)

    features.append(np.mean(zcr))
    features.append(np.std(zcr))

    # --------------------------------------------------
    # 9. RMS Energy
    # --------------------------------------------------
    rms = librosa.feature.rms(y=y)

    features.append(np.mean(rms))
    features.append(np.std(rms))

    return np.array(features, dtype=np.float32)


print("Feature extraction function created successfully.")

In [ ]:
test_features = extract_audio_features(
    train.loc[0, "audio_path"]
)

print("Number of extracted features:", len(test_features))
print("Feature vector shape:", test_features.shape)
print("First 10 features:", test_features[:10])

In [ ]:
X_train_features = []

print("Extracting features from training audio...")

for audio_path in tqdm(
    train["audio_path"],
    total=len(train)
):
    
    features = extract_audio_features(audio_path)
    X_train_features.append(features)

X_train_features = np.array(
    X_train_features,
    dtype=np.float32
)

print("\nTraining feature matrix shape:")
print(X_train_features.shape)

In [ ]:
X_test_features = []

print("Extracting features from test audio...")

for audio_path in tqdm(
    test["audio_path"],
    total=len(test)
):
    
    features = extract_audio_features(audio_path)
    X_test_features.append(features)

X_test_features = np.array(
    X_test_features,
    dtype=np.float32
)

print("\nTest feature matrix shape:")
print(X_test_features.shape)

In [ ]:
print("Training shape:", X_train_features.shape)
print("Test shape:", X_test_features.shape)

print(
    "NaN values in training:",
    np.isnan(X_train_features).sum()
)

print(
    "NaN values in test:",
    np.isnan(X_test_features).sum()
)

print(
    "Infinite values in training:",
    np.isinf(X_train_features).sum()
)

print(
    "Infinite values in test:",
    np.isinf(X_test_features).sum()
)

In [ ]:
np.save(
    "/kaggle/working/X_train_features.npy",
    X_train_features
)

np.save(
    "/kaggle/working/X_test_features.npy",
    X_test_features
)

np.save(
    "/kaggle/working/y_train.npy",
    train["label"].values
)

print("Feature files saved successfully.")

In [ ]:
X = X_train_features
y = train["label"].values.astype(np.float32)

X_test = X_test_features

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Number of features:", X_train.shape[1])

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

extra_trees = ExtraTreesRegressor(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("Training Extra Trees...")

extra_trees.fit(
    X_train,
    y_train
)

print("Training completed!")

In [ ]:
import numpy as np

val_pred_et = extra_trees.predict(X_val)

# Grammar score must be between 0 and 5
val_pred_et = np.clip(
    val_pred_et,
    0,
    5
)

print("First 10 predictions:")
print(val_pred_et[:10])

In [ ]:
from sklearn.metrics import mean_squared_error

rmse_et = np.sqrt(
    mean_squared_error(
        y_val,
        val_pred_et
    )
)

print("Extra Trees Validation RMSE:", rmse_et)

In [ ]:
from scipy.stats import pearsonr

pearson_et = pearsonr(
    y_val,
    val_pred_et
)[0]

print("Extra Trees Pearson Correlation:", pearson_et)

In [ ]:
train_pred_et = extra_trees.predict(X_train)

train_pred_et = np.clip(
    train_pred_et,
    0,
    5
)

train_rmse_et = np.sqrt(
    mean_squared_error(
        y_train,
        train_pred_et
    )
)

print("Extra Trees Training RMSE:", train_rmse_et)
print("Extra Trees Validation RMSE:", rmse_et)
print("Extra Trees Validation Pearson:", pearson_et)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))

plt.scatter(
    y_val,
    val_pred_et,
    alpha=0.7
)

plt.plot(
    [0, 5],
    [0, 5],
    linestyle="--"
)

plt.xlabel("Actual Grammar Score")
plt.ylabel("Predicted Grammar Score")
plt.title("Extra Trees: Actual vs Predicted")

plt.xlim(0, 5)
plt.ylim(0, 5)

plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

random_forest = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

random_forest.fit(
    X_train,
    y_train
)

print("Training completed!")

In [ ]:
val_pred_rf = random_forest.predict(X_val)

val_pred_rf = np.clip(
    val_pred_rf,
    0,
    5
)

rmse_rf = np.sqrt(
    mean_squared_error(
        y_val,
        val_pred_rf
    )
)

pearson_rf = pearsonr(
    y_val,
    val_pred_rf
)[0]

train_pred_rf = random_forest.predict(X_train)

train_pred_rf = np.clip(
    train_pred_rf,
    0,
    5
)

train_rmse_rf = np.sqrt(
    mean_squared_error(
        y_train,
        train_pred_rf
    )
)

print("Random Forest Training RMSE:", train_rmse_rf)
print("Random Forest Validation RMSE:", rmse_rf)
print("Random Forest Pearson:", pearson_rf)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_samples_leaf=3,
    random_state=42
)

print("Training Gradient Boosting...")

gradient_boosting.fit(
    X_train,
    y_train
)

print("Training completed!")

In [ ]:
val_pred_gb = gradient_boosting.predict(X_val)

val_pred_gb = np.clip(
    val_pred_gb,
    0,
    5
)

rmse_gb = np.sqrt(
    mean_squared_error(
        y_val,
        val_pred_gb
    )
)

pearson_gb = pearsonr(
    y_val,
    val_pred_gb
)[0]

train_pred_gb = gradient_boosting.predict(X_train)

train_pred_gb = np.clip(
    train_pred_gb,
    0,
    5
)

train_rmse_gb = np.sqrt(
    mean_squared_error(
        y_train,
        train_pred_gb
    )
)

print("Gradient Boosting Training RMSE:", train_rmse_gb)
print("Gradient Boosting Validation RMSE:", rmse_gb)
print("Gradient Boosting Pearson:", pearson_gb)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Extra Trees",
        "Random Forest",
        "Gradient Boosting"
    ],
    
    "Training RMSE": [
        train_rmse_et,
        train_rmse_rf,
        train_rmse_gb
    ],
    
    "Validation RMSE": [
        rmse_et,
        rmse_rf,
        rmse_gb
    ],
    
    "Validation Pearson": [
        pearson_et,
        pearson_rf,
        pearson_gb
    ]
})

results

In [ ]:
results.sort_values(
    by="Validation RMSE",
    ascending=True
)

In [ ]:
results.sort_values(
    by="Validation Pearson",
    ascending=False
)

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost training completed!")

In [ ]:
val_pred_xgb = xgb_model.predict(X_val)

val_pred_xgb = np.clip(
    val_pred_xgb,
    0,
    5
)

rmse_xgb = np.sqrt(
    mean_squared_error(
        y_val,
        val_pred_xgb
    )
)

pearson_xgb = pearsonr(
    y_val,
    val_pred_xgb
)[0]

train_pred_xgb = xgb_model.predict(X_train)

train_pred_xgb = np.clip(
    train_pred_xgb,
    0,
    5
)

train_rmse_xgb = np.sqrt(
    mean_squared_error(
        y_train,
        train_pred_xgb
    )
)

print("XGBoost Training RMSE:", train_rmse_xgb)
print("XGBoost Validation RMSE:", rmse_xgb)
print("XGBoost Pearson:", pearson_xgb)

In [ ]:
results = pd.concat([
    results,
    pd.DataFrame({
        "Model": ["XGBoost"],
        "Training RMSE": [train_rmse_xgb],
        "Validation RMSE": [rmse_xgb],
        "Validation Pearson": [pearson_xgb]
    })
], ignore_index=True)

display(results.sort_values("Validation RMSE"))

In [ ]:
ensemble_pred = (
    0.4 * val_pred_xgb +
    0.3 * val_pred_et +
    0.3 * val_pred_gb
)

ensemble_pred = np.clip(
    ensemble_pred,
    0,
    5
)

ensemble_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        ensemble_pred
    )
)

ensemble_pearson = pearsonr(
    y_val,
    ensemble_pred
)[0]

print("Ensemble Validation RMSE:", ensemble_rmse)
print("Ensemble Pearson:", ensemble_pearson)

In [ ]:
print("Current results")
print("-------------------------")

print("Extra Trees:")
print("RMSE:", rmse_et)
print("Pearson:", pearson_et)

print("\nRandom Forest:")
print("RMSE:", rmse_rf)
print("Pearson:", pearson_rf)

print("\nGradient Boosting:")
print("RMSE:", rmse_gb)
print("Pearson:", pearson_gb)

print("\nXGBoost:")
print("RMSE:", rmse_xgb)
print("Pearson:", pearson_xgb)

print("\nEnsemble:")
print("RMSE:", ensemble_rmse)
print("Pearson:", ensemble_pearson)


In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_val,
    ensemble_pred,
    alpha=0.7
)

plt.plot(
    [0, 5],
    [0, 5],
    linestyle="--"
)

plt.xlabel("Actual Grammar Score")
plt.ylabel("Predicted Grammar Score")

plt.title(
    f"Ensemble Predictions\n"
    f"RMSE={ensemble_rmse:.4f}, "
    f"Pearson={ensemble_pearson:.4f}"
)

plt.xlim(0, 5)
plt.ylim(0, 5)

plt.show()

In [ ]:
print("Training vs Validation")
print("-----------------------------")

print(
    "XGBoost:",
    train_rmse_xgb,
    "→",
    rmse_xgb
)

print(
    "Extra Trees:",
    train_rmse_et,
    "→",
    rmse_et
)

print(
    "Random Forest:",
    train_rmse_rf,
    "→",
    rmse_rf
)

print(
    "Gradient Boosting:",
    train_rmse_gb,
    "→",
    rmse_gb
)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Extra Trees": extra_trees,
    "Random Forest": random_forest,
    "Gradient Boosting": gradient_boosting,
    "XGBoost": xgb_model
}

cv_results = []

for model_name, base_model in models.items():

    rmse_scores = []
    pearson_scores = []

    print(f"\nRunning CV for: {model_name}")

    for fold, (train_idx, val_idx) in enumerate(
        kf.split(X),
        start=1
    ):

        X_tr = X[train_idx]
        X_va = X[val_idx]

        y_tr = y[train_idx]
        y_va = y[val_idx]

        model = clone(base_model)

        model.fit(X_tr, y_tr)

        pred = model.predict(X_va)

        pred = np.clip(
            pred,
            0,
            5
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_va,
                pred
            )
        )

        pearson = pearsonr(
            y_va,
            pred
        )[0]

        rmse_scores.append(rmse)
        pearson_scores.append(pearson)

        print(
            f"Fold {fold}: "
            f"RMSE={rmse:.4f}, "
            f"Pearson={pearson:.4f}"
        )

    cv_results.append({
        "Model": model_name,
        "Mean RMSE": np.mean(rmse_scores),
        "Std RMSE": np.std(rmse_scores),
        "Mean Pearson": np.mean(pearson_scores),
        "Std Pearson": np.std(pearson_scores)
    })

cv_results = pd.DataFrame(cv_results)

display(
    cv_results.sort_values(
        "Mean RMSE"
    )
)

In [ ]:
final_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X,
    y
)

print("Final model trained on all 769 samples.")

In [ ]:
test_predictions = final_model.predict(
    X_test
)

test_predictions = np.clip(
    test_predictions,
    0,
    5
)

print("Number of predictions:", len(test_predictions))
print("First 20 predictions:")
print(test_predictions[:20])

print("\nMinimum prediction:", test_predictions.min())
print("Maximum prediction:", test_predictions.max())
print("Average prediction:", test_predictions.mean())

In [ ]:
submission = sample_submission.copy()

submission["label"] = test_predictions

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("Submission created successfully!")

display(submission.head(10))

In [ ]:
print("Train rows:", len(train))
print("Test rows:", len(test))
print("Sample submission rows:", len(sample_submission))

print("\nTest columns:")
print(test.columns.tolist())

print("\nSample submission columns:")
print(sample_submission.columns.tolist())

In [ ]:
print("Test filenames:", len(test["filename"]))
print("Unique test filenames:", test["filename"].nunique())

print("Submission filenames:", len(sample_submission["filename"]))
print("Unique submission filenames:", sample_submission["filename"].nunique())

In [ ]:
missing_from_submission = set(test["filename"]) - set(
    sample_submission["filename"]
)

extra_in_submission = set(sample_submission["filename"]) - set(
    test["filename"]
)

print("Test files missing from sample submission:",
      len(missing_from_submission))

print("Extra files in sample submission:",
      len(extra_in_submission))

print("\nMissing filenames:")
print(list(missing_from_submission)[:20])

In [ ]:
print("TEST:")
display(test.head(20))

print("\nSAMPLE SUBMISSION:")
display(sample_submission.head(20))

print("\nLast 20 TEST filenames:")
print(test["filename"].tail(20).tolist())

print("\nLast 20 SAMPLE SUBMISSION filenames:")
print(sample_submission["filename"].tail(20).tolist())

In [ ]:
common_files = sorted(
    set(test["filename"]) &
    set(sample_submission["filename"])
)

print("Number of common filenames:", len(common_files))

print("\nCommon filenames:")
print(common_files)

In [ ]:
print("TEST CSV")
display(test.head(10))
print("Test shape:", test.shape)

print("\nSAMPLE SUBMISSION CSV")
display(sample_submission.head(10))
print("Sample submission shape:", sample_submission.shape)

print("\nTEST last 10:")
display(test.tail(10))

print("\nSAMPLE SUBMISSION last 10:")
display(sample_submission.tail(10))

In [ ]:
# Check the filename ranges and patterns

print("TEST filename examples:")
print(test["filename"].tolist()[:30])

print("\nSAMPLE SUBMISSION filename examples:")
print(sample_submission["filename"].tolist()[:30])

print("\nNumber of unique TEST filenames:",
      test["filename"].nunique())

print("Number of unique SAMPLE filenames:",
      sample_submission["filename"].nunique())

# Check whether test filenames are audio_0.wav to audio_215.wav
test_numbers = (
    test["filename"]
    .str.extract(r"audio_(\d+)\.wav")[0]
    .astype(int)
)

sample_numbers = (
    sample_submission["filename"]
    .str.extract(r"audio_(\d+)\.wav")[0]
    .astype(int)
)

print("\nTEST number range:",
      test_numbers.min(), "to", test_numbers.max())

print("SAMPLE SUBMISSION number range:",
      sample_numbers.min(), "to", sample_numbers.max())

In [ ]:
# Create submission using the actual test.csv order

submission = pd.DataFrame({
    "filename": test["filename"],
    "label": test_predictions
})

print("Submission shape:", submission.shape)
print("\nFirst 10 rows:")
display(submission.head(10))

print("\nLast 10 rows:")
display(submission.tail(10))

print("\nPrediction statistics:")
print(submission["label"].describe())

In [ ]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission saved successfully!")
print("File: /kaggle/working/submission.csv")
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())

In [ ]:
# Calculate final training metrics

final_train_pred = final_model.predict(X)
final_train_pred = np.clip(final_train_pred, 0, 5)

final_train_rmse = np.sqrt(
    mean_squared_error(y, final_train_pred)
)

final_train_pearson = pearsonr(
    y, final_train_pred
)[0]

print("Final Training RMSE:", final_train_rmse)
print("Final Training Pearson Correlation:", final_train_pearson)

# Model Evaluation and Results

## Model Comparison

Four regression models were evaluated using 5-fold cross-validation:

| Model | Mean RMSE | Mean Pearson Correlation |
|---|---:|---:|
| XGBoost | 0.7759 | 0.7759 |
| Extra Trees | 0.7875 | 0.7712 |
| Gradient Boosting | 0.7998 | 0.7602 |
| Random Forest | 0.7999 | 0.7618 |

XGBoost achieved the best overall cross-validation performance and was selected as the final model.

## Final Model

The final XGBoost regressor was trained on all 769 available training samples and used to generate predictions for the 216 test audio samples.

## Required Training Performance

The final model achieved:

- **Training RMSE:** 0.2529
- **Training Pearson Correlation:** 0.9830

The training metrics are reported separately from cross-validation metrics because they measure performance on the same data used to train the final model.

## Cross-Validation Performance

The more reliable estimate of generalization performance comes from 5-fold cross-validation:

- **Mean RMSE:** 0.7759
- **Mean Pearson Correlation:** 0.7759

These results indicate that the model captures meaningful relationships between acoustic features and grammar scores while leaving room for further improvement with advanced speech and language representations.

# Conclusion

This project developed an end-to-end Grammar Scoring Engine for spoken English audio.

The pipeline included:

1. Loading and exploring the SHL spoken-audio dataset.
2. Visualizing waveforms, spectrograms, and MFCC features.
3. Extracting MFCC, delta-MFCC, Mel-spectrogram, chroma, spectral, zero-crossing-rate, and RMS features.
4. Training and comparing multiple regression models.
5. Using 5-fold cross-validation for model evaluation.
6. Selecting XGBoost as the best-performing baseline model.
7. Training the final model on all 769 training samples.
8. Generating grammar-score predictions for 216 test audio samples.
9. Saving the final predictions to `submission.csv`.

The final submission contains 216 predictions with grammar scores constrained to the required 0–5 range.

The approach provides a strong traditional machine-learning baseline. Future improvements could include pretrained speech representations such as wav2vec 2.0, HuBERT, Whisper embeddings, and additional linguistic features.